In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os

# ===== 1. 诊断数据结构 =====
file_path = '/Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/data/2025JanFeb.nc'
ds = xr.open_dataset(file_path)

print("=== 数据集基本信息 ===")
print(f"维度: {list(ds.dims)}")
print(f"坐标: {list(ds.coords)}")
print(f"变量: {list(ds.data_vars)}")
print(f"\n时间范围: {ds.valid_time.values[0]} 至 {ds.valid_time.values[-1]}")
print(f"时间数量: {len(ds.valid_time)}")
print(f"气压层: {ds.pressure_level.values}")
print(f"纬度范围: {float(ds.latitude.min()):.1f} ~ {float(ds.latitude.max()):.1f}")
print(f"经度范围: {float(ds.longitude.min()):.1f} ~ {float(ds.longitude.max()):.1f}")

# ===== 2. 提取500hPa位势高度 =====
z_500 = ds['z'].sel(pressure_level=500.0) / 9.80665  # 转为位势米
lat = ds.latitude.values
lon = ds.longitude.values
times = ds.valid_time.values

print(f"\n500hPa位势高度场形状: {z_500.shape}")
print(f"将生成 {len(times)} 张图")

# ===== 3. 创建输出目录 =====
output_dir = '/Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/figures/case selection'
os.makedirs(output_dir, exist_ok=True)

# ===== 4. 批量绘制每日500hPa位势高度场 =====
print("\n开始绘图...")

for i, time in enumerate(times):
    # 创建画布
    fig, ax = plt.subplots(figsize=(14, 9), subplot_kw={'projection': ccrs.PlateCarree()})
    
    # 地图要素
    ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3)
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle='--', alpha=0.5)
    
    # 网格线
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    
    # 提取当前时次数据
    z_now = z_500.isel(valid_time=i)
    
    # 位势高度填色
    cf = ax.contourf(lon, lat, z_now, levels=np.arange(5100, 6000, 30), 
                      cmap='RdBu_r', extend='both', alpha=0.6)
    
    # 位势高度等值线
    contour = ax.contour(lon, lat, z_now, levels=np.arange(5200, 6000, 40), 
                          colors='black', linewidths=1.2)
    ax.clabel(contour, inline=True, fontsize=8, fmt='%d')
    
    # 设置显示范围
    ax.set_extent([70, 150, 20, 60], crs=ccrs.PlateCarree())
    
    # 标题和colorbar
    time_str = np.datetime_as_string(time, unit='D')
    ax.set_title(f'ERA5 500 hPa Geopotential Height\n{time_str} 00:00 UTC', 
                 fontsize=14, fontweight='bold')
    
    cbar = plt.colorbar(cf, ax=ax, orientation='vertical', shrink=0.8, pad=0.02)
    cbar.set_label('Geopotential Height (gpm)', fontsize=12)
    
    # 保存
    filename = f"{np.datetime_as_string(time, unit='D')}_z500.png"
    filepath = os.path.join(output_dir, filename)
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.close()
    
    if (i+1) % 10 == 0:
        print(f"  已完成 {i+1}/{len(times)} 张图")

print(f"\n全部完成！共 {len(times)} 张图保存在 {output_dir}")

=== 数据集基本信息 ===
维度: ['valid_time', 'pressure_level', 'latitude', 'longitude']
坐标: ['number', 'valid_time', 'pressure_level', 'latitude', 'longitude', 'expver']
变量: ['z', 'u', 'v']

时间范围: 2025-01-01T00:00:00.000000000 至 2025-02-28T23:00:00.000000000
时间数量: 1416
气压层: [500.]
纬度范围: -90.0 ~ 90.0
经度范围: 0.0 ~ 359.8

500hPa位势高度场形状: (1416, 721, 1440)
将生成 1416 张图

开始绘图...
  已完成 10/1416 张图
  已完成 20/1416 张图


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os
from matplotlib.colors import Normalize

# ===== 1. 读取数据 =====
file_path = '/Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/data/2025JanFeb.nc'
ds = xr.open_dataset(file_path)

# 提取500hPa位势高度并转换为位势米
z_500 = ds['z'].sel(pressure_level=500.0) / 9.80665
lat = ds.latitude.values
lon = ds.longitude.values
times = ds.valid_time.values

print(f"数据信息:")
print(f"  时间数量: {len(times)}")
print(f"  500hPa位势高度场形状: {z_500.shape}")

# ===== 2. 设置总图参数 =====
n_rows, n_cols = 8, 8  # 8x8布局，共64个子图
n_subplots = n_rows * n_cols

# 检查时间数量
if len(times) < n_subplots:
    print(f"警告：只有 {len(times)} 个时次，少于 {n_subplots} 个")
    n_plots = len(times)
else:
    n_plots = n_subplots
    times = times[:n_plots]  # 只取前64个时次

print(f"将绘制 {n_plots} 张图（{n_rows}x{n_cols}）")

# ===== 3. 创建大图 =====
fig = plt.figure(figsize=(30, 30))  # 大画布

# 统一色阶范围
z_min, z_max = 5000, 5900
levels_fill = np.arange(z_min, z_max, 20)
levels_contour = np.arange(z_min, z_max, 40)

# 创建投影对象（用于所有子图）
proj = ccrs.PlateCarree()

# ===== 4. 逐个绘制子图 =====
for i, time in enumerate(times):
    # 创建子图
    ax = fig.add_subplot(n_rows, n_cols, i+1, projection=proj)
    
    # 提取当前时次数据
    z_now = z_500.isel(valid_time=i)
    
    # 绘制地图要素
    ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.3, linewidth=0.3)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle='--', alpha=0.5)
    
    # 位势高度填色
    cf = ax.contourf(lon, lat, z_now, levels=levels_fill, 
                      cmap='RdBu_r', extend='both')
    
    # 位势高度等值线
    contour = ax.contour(lon, lat, z_now, levels=levels_contour, 
                          colors='black', linewidths=0.8)
    ax.clabel(contour, inline=True, fontsize=6, fmt='%d')
    
    # 设置显示范围（可根据需要调整）
    ax.set_extent([70, 150, 20, 60], crs=ccrs.PlateCarree())
    
    # 添加网格线（只有部分子图显示标签以避免重叠）
    if i % n_cols == 0:  # 最左边一列显示经度标签
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', 
                         alpha=0.5, linestyle='--', dms=True)
        gl.right_labels = False
        gl.top_labels = False
    elif i < n_cols:  # 最上面一行显示纬度标签
        gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', 
                         alpha=0.5, linestyle='--', dms=True)
        gl.right_labels = False
        gl.bottom_labels = False
    else:
        gl = ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', 
                         alpha=0.5, linestyle='--')
    
    # 添加标题（日期）
    time_str = np.datetime_as_string(time, unit='D')
    ax.set_title(f'{time_str}', fontsize=10, fontweight='bold', pad=3)

# ===== 5. 添加总标题和colorbar =====
fig.suptitle('ERA5 500 hPa Geopotential Height (00:00 UTC)\nMarch-April 2025', 
             fontsize=20, fontweight='bold', y=0.98)

# 添加colorbar
cbar_ax = fig.add_axes([0.92, 0.25, 0.02, 0.5])  # [left, bottom, width, height]
cbar = fig.colorbar(cf, cax=cbar_ax, orientation='vertical')
cbar.set_label('Geopotential Height (gpm)', fontsize=16)
cbar.ax.tick_params(labelsize=12)

# 调整子图间距
plt.subplots_adjust(left=0.05, right=0.9, top=0.95, bottom=0.05, 
                    wspace=0.1, hspace=0.15)

# ===== 6. 保存大图 =====
output_dir = '/Users/weiyingwan/Desktop/校内课程/大三下/数值天气预报/project/figures/case selection/summary'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "z500_composite_8x8.png")
plt.savefig(output_path, dpi=200, bbox_inches='tight')
print(f"\n大图已保存至: {output_path}")

# 可选：同时保存PDF格式以便打印
pdf_path = os.path.join(output_dir, "z500_composite_8x8.pdf")
plt.savefig(pdf_path, bbox_inches='tight')
print(f"PDF版本已保存至: {pdf_path}")

plt.close()
print("完成！")